In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile
import os

ZIP_PATH = 'drive/MyDrive/Task2.zip'

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall("projem")
os.chdir("projem")

In [1]:
import sys
sys.path.append('..')
from Non_local_gnn_model import GraphTransformerNet

In [2]:
import sys
sys.path.append('projem/Task2')

from preprocess import JetEventDataset
import torch
import pickle
from torch_geometric.transforms import NormalizeFeatures
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader

import torch.nn as nn

In [3]:
import random
import numpy as np

def set_all_seeds(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    g = torch.Generator()
    g.manual_seed(seed)

    return g

In [4]:
ROOT_DIR = 'projem/Task2/Dataset/Data'
g = set_all_seeds(42)
VAL_SPLIT = 0.20
TEST_SPLIT = 0.10
BATCH_SIZE = 128
NUM_WORKERS = 6
DEVICE = 'cuda'

In [5]:
dataset = JetEventDataset(root=ROOT_DIR, pre_transform=NormalizeFeatures(), k=5)

labels = [int(data.y) for data in dataset]
indices = list(range(len(dataset)))

train_indices, test_indices, _, _ = train_test_split(indices, labels, test_size=TEST_SPLIT, random_state=42, stratify=labels)
train_indices, val_indices, _, _ = train_test_split(train_indices, [labels[i] for i in train_indices], test_size=VAL_SPLIT, random_state=42, stratify=[labels[i] for i in train_indices])

train_dataset = Subset(dataset, train_indices)
val_dataset   = Subset(dataset, val_indices)
test_dataset  = Subset(dataset, test_indices)


In [6]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, generator=g)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, generator=g)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, generator=g)

In [7]:
labels_count = {}
for label in labels:
    if label not in labels_count:
        labels_count[label] = 0
    labels_count[label] += 1

print(f"Class distribution: {labels_count}")

Class distribution: {0: 69653, 1: 69653}


In [8]:
in_dim     = 3
hidden     = 128
layers     = 3
heads      = 2
edge_dim   = None
pe_dim     = 0
dropout    = 0.2
droppath   = 0.1
num_classes = 1
task_type   = "binary"
readout     = "attn"

model = GraphTransformerNet(
    in_dim=in_dim,
    hidden=hidden,
    layers=layers,
    heads=heads,
    edge_dim=edge_dim,
    pe_dim=pe_dim,
    dropout=dropout,
    droppath=droppath,
    num_classes=num_classes,
    task_type=task_type,
    readout=readout
)

print(model)


GraphTransformerNet(
  (input_proj): Linear(in_features=3, out_features=128, bias=True)
  (blocks): ModuleList(
    (0-2): 3 x GraphTransformerBlock(
      (pre_attn): PreNorm(
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (attn): TransformerConv(128, 64, heads=2)
      (res_proj): Identity()
      (drop_path1): DropPath()
      (drop_path2): DropPath()
      (ffn): Sequential(
        (0): PreNorm(
          (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (1): Linear(in_features=128, out_features=512, bias=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.2, inplace=False)
        (4): Linear(in_features=512, out_features=128, bias=True)
      )
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (readout): GlobalAttention(gate_nn=Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=64, out_features=1, bias=Tr

/usr/local/lib/python3.12/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  warnings.warn(out)


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheculer = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [10]:
from projem.Task2.train_gnnModel import train_model
best_value = train_model(model, train_loader, val_loader, DEVICE, criterion, optimizer, early_stopping_patience=10, EPOCHS=100, scheduler=scheculer)

INFO:app_logger:Starting training for 100 epochs
INFO:app_logger:
Epoch 1/100
INFO:app_logger:---------------


Epoch 1/100 - Training:   0%|          | 0/784 [00:00<?, ?it/s]

INFO:app_logger:Batch 10: Loss = 0.7135
INFO:app_logger:Batch 20: Loss = 0.6959
INFO:app_logger:Batch 30: Loss = 0.6925
INFO:app_logger:Batch 40: Loss = 0.6942
INFO:app_logger:Batch 50: Loss = 0.6910
INFO:app_logger:Batch 60: Loss = 0.6906
INFO:app_logger:Batch 70: Loss = 0.6961
INFO:app_logger:Batch 80: Loss = 0.6894
INFO:app_logger:Batch 90: Loss = 0.6922
INFO:app_logger:Batch 100: Loss = 0.6933
INFO:app_logger:Batch 110: Loss = 0.6886
INFO:app_logger:Batch 120: Loss = 0.6926
INFO:app_logger:Batch 130: Loss = 0.7136
INFO:app_logger:Batch 140: Loss = 0.6920
INFO:app_logger:Batch 150: Loss = 0.6967
INFO:app_logger:Batch 160: Loss = 0.6922
INFO:app_logger:Batch 170: Loss = 0.7061
INFO:app_logger:Batch 180: Loss = 0.6943
INFO:app_logger:Batch 190: Loss = 0.6956
INFO:app_logger:Batch 200: Loss = 0.6884
INFO:app_logger:Batch 210: Loss = 0.6909
INFO:app_logger:Batch 220: Loss = 0.6941
INFO:app_logger:Batch 230: Loss = 0.6914
INFO:app_logger:Batch 240: Loss = 0.6930
INFO:app_logger:Batch 250

Epoch 1/100 - Validation:   0%|          | 0/196 [00:00<?, ?it/s]

INFO:app_logger:Current Learning Rate: 0.001000
INFO:app_logger:Epoch 1 completed - Train Loss: 0.6939, Train_Acc: 0.5012, Train_AUC: 0.5015, F1_train: 0.5010, Val Loss: 0.6932, Val_F1: 0.3333, Val_Acc: 0.5000, Val_AUC: 0.7088
INFO:app_logger:Confusion Matrix:
[[12537, 0], [12538, 0]]
TP: 0, TN: 12537, FP: 0, FN: 12538 | Precision: 0.0000, Recall: 0.0000, Specificity: 1.0000
INFO:app_logger: Validation loss improved. Model saved to best_model.pth
INFO:app_logger:
Epoch 2/100
INFO:app_logger:---------------


...

Epoch 13/100 - Training:   0%|          | 0/784 [00:00<?, ?it/s]

INFO:app_logger:Batch 10: Loss = 0.6820
INFO:app_logger:Batch 20: Loss = 0.6890
INFO:app_logger:Batch 30: Loss = 0.6853
INFO:app_logger:Batch 40: Loss = 0.6959
INFO:app_logger:Batch 50: Loss = 0.6901
INFO:app_logger:Batch 60: Loss = 0.6889
INFO:app_logger:Batch 70: Loss = 0.6901
INFO:app_logger:Batch 80: Loss = 0.6784
INFO:app_logger:Batch 90: Loss = 0.6967
INFO:app_logger:Batch 100: Loss = 0.6984
INFO:app_logger:Batch 110: Loss = 0.6941
INFO:app_logger:Batch 120: Loss = 0.6935
INFO:app_logger:Batch 130: Loss = 0.6944
INFO:app_logger:Batch 140: Loss = 0.6803
INFO:app_logger:Batch 150: Loss = 0.6900
INFO:app_logger:Batch 160: Loss = 0.6890
INFO:app_logger:Batch 170: Loss = 0.6916
INFO:app_logger:Batch 180: Loss = 0.7009
INFO:app_logger:Batch 190: Loss = 0.6985
INFO:app_logger:Batch 200: Loss = 0.6913
INFO:app_logger:Batch 210: Loss = 0.6942
INFO:app_logger:Batch 220: Loss = 0.6928
INFO:app_logger:Batch 230: Loss = 0.6920
INFO:app_logger:Batch 240: Loss = 0.6816
INFO:app_logger:Batch 250

Epoch 13/100 - Validation:   0%|          | 0/196 [00:00<?, ?it/s]

INFO:app_logger:Current Learning Rate: 0.000250
INFO:app_logger:Epoch 13 completed - Train Loss: 0.6913, Train_Acc: 0.5240, Train_AUC: 0.5321, F1_train: 0.5157, Val Loss: 1.0120, Val_F1: 0.3333, Val_Acc: 0.5000, Val_AUC: 0.3020
INFO:app_logger:Confusion Matrix:
[[12537, 0], [12538, 0]]
TP: 0, TN: 12537, FP: 0, FN: 12538 | Precision: 0.0000, Recall: 0.0000, Specificity: 1.0000
INFO:app_logger: No improvement. Early stopping counter: 10/10
INFO:app_logger: Early stopping triggered. Stopping training.
INFO:app_logger:Best epoch so far: Epoch 3 completed - Train Loss: 0.6932, Train_Acc: 0.5004, Train_AUC: 0.5012, F1_train: 0.4997, Val Loss: 0.6931, Val_F1: 0.3333, Val_Acc: 0.5000, Val_AUC: 0.6993
Confusion Matrix:
[[12537, 0], [12538, 0]]
TP: 0, TN: 12537, FP: 0, FN: 12538 | Precision: 0.0000, Recall: 0.0000, Specificity: 1.0000



Training completed!
